# Bayesian linear regression

_Classical ML_

**Don't fit one line — keep a whole distribution of plausible lines.**

Ordinary linear regression returns one best line. But with little data, many lines fit almost as well.

     Bayesian linear regression keeps all of them, weighted by how plausible each is.

---

This notebook is a practice scaffold for the **Bayesian linear regression** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

We'll fit a **BayesianRidge** model on a small synthetic regression problem where we know the true coefficients. We do it in three steps: (1) make the data, (2) fit the model and inspect what it learned, (3) predict *with uncertainty*.

### Step 1 — Make a synthetic regression dataset

`make_regression` builds a problem from a known linear rule plus noise, and hands back the **true coefficients** so we can check our fit later. With only 120 samples and added noise, no estimator can recover the coefficients perfectly — which is exactly where keeping track of uncertainty pays off.

In [ ]:
from sklearn.datasets import make_regression

X, y, coef = make_regression(
    n_samples=120,
    n_features=4,
    noise=8.0,
    coef=True,        # also return the true coefficients
    random_state=0,
)

### Step 2 — Fit BayesianRidge and inspect what it learned

BayesianRidge doesn't return a single weight vector — it infers a *posterior distribution* over the weights. Its `coef_` are the posterior **means**, and it also estimates two precisions: `alpha_` (how much observation noise it thinks there is) and `lambda_` (how tightly the weights are regularised). Compare the learned means to the true coefficients printed alongside.

In [ ]:
from sklearn.linear_model import BayesianRidge

br = BayesianRidge(compute_score=True)
br.fit(X, y)

print("true coefficients :", np.round(coef, 2))
print("learned coef (mean):", np.round(br.coef_, 2))
print("intercept:", round(br.intercept_, 3))
print("alpha (noise precision):", round(br.alpha_, 4))
print("lambda (weight precision):", round(br.lambda_, 4))

### Step 3 — Predict with uncertainty

The real payoff of the Bayesian view: every prediction comes with an error bar. Passing `return_std=True` gives both the predicted mean and its standard deviation, so each prediction reads as `mean ± std`. Wider bars mean the model is less sure about that point.

In [ ]:
mean, std = br.predict(X[:3], return_std=True)

for i in range(3):
    print("pred %.2f +/- %.2f   (true %.2f)" % (mean[i], std[i], y[i]))

## Visualize the data & results

_Fitting diabetes progression from BMI, how does the Bayesian posterior slope compare to plain OLS?_

### Step 1 — Fit both models on a single real feature

We switch to the real **diabetes** dataset and use just the BMI column so the fit is a line we can draw. We fit two models on the same data: ordinary least squares (`LinearRegression`) and `BayesianRidge`. Printing their slopes shows how close the two estimates of the BMI effect are.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import BayesianRidge, LinearRegression

dia = load_diabetes()
x = dia.data[:, 2]            # BMI feature
y = dia.target.astype(float)
X = x.reshape(-1, 1)

ols = LinearRegression().fit(X, y)
br = BayesianRidge().fit(X, y)

print(ols.coef_[0], br.coef_[0])

### Step 2 — Plot the OLS fit, posterior mean, and prior mean

We draw the scatter of data plus three reference lines. The OLS fit and the Bayesian posterior mean nearly overlap with this much data — the prior barely tugs them apart. The flat **prior mean** line (the target's average) is where the Bayesian model would sit with *no* evidence; the data pulls it toward the OLS slope.

In [ ]:
xs = np.linspace(x.min() - 0.01, x.max() + 0.01, 30).reshape(-1, 1)

plt.scatter(x, y, c="#ffb454", s=30)
plt.plot(xs.ravel(), ols.predict(xs), c="#ff7b72", linestyle="--",
         label="OLS fit")
plt.plot(xs.ravel(), br.predict(xs), c="#4ea1ff",
         label="posterior mean")
plt.plot(xs.ravel(), np.full_like(xs.ravel(), y.mean()), c="#9aa7b4",
         linestyle="--", label="prior mean")
plt.legend()
plt.title("Bayesian ridge vs OLS on Diabetes (BMI)")
plt.xlabel("BMI (mean-centered, scaled)")
plt.ylabel("disease progression")
plt.show()